In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from xgboost import XGBClassifier

# ----------------------------
# 1. LOAD TRAINING DATA
# ----------------------------
train_path = "https://raw.githubusercontent.com/marcojs253-crypto/P_2/refs/heads/main/Data/TrainingData.csv"
df_train = pd.read_csv(train_path)

# Fjern irrelevant features
irrelevant_cols = ["filnavn", "beta", "snr_db"]  # fjern beta og snr fra features
for col in irrelevant_cols:
    if col in df_train.columns:
        df_train = df_train.drop(columns=[col])

# ----------------------------
# 2. FEATURES & TARGET
# ----------------------------
X = df_train.drop(columns=["target"])
y = df_train["target"]

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Klasse mapping:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name} -> {i}")

# ----------------------------
# 3. 5-FOLD CV + ÆGTE GRID SEARCH PR. FEATURE (XGBOOST)
# ----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Mindre grid så ægte grid search stadig er realistisk pr. feature
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

grid_combinations = (
    len(param_grid["n_estimators"]) *
    len(param_grid["max_depth"]) *
    len(param_grid["learning_rate"]) *
    len(param_grid["subsample"]) *
    len(param_grid["colsample_bytree"])
    )
print(f"Grid-kombinationer pr. feature: {grid_combinations}")
print(f"Model fits pr. feature (kombinationer x 5 folds): {grid_combinations * 5}")

single_feature_tuning = []

for feature in X.columns:
    X_one = X[[feature]]
    print(f"\nFeature: {feature}")
    base_model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        random_state=42,
        eval_metric="mlogloss",
        tree_method="hist"
    )

    search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        verbose=0
    )

    search.fit(X_one, y_encoded)
    
    # Udskriv accuracy for hver fold
    print("Fold accuracies:", search.cv_results_["split0_test_score"], search.cv_results_["split1_test_score"], search.cv_results_["split2_test_score"], search.cv_results_["split3_test_score"], search.cv_results_["split4_test_score"])
    print(f"Best CV accuracy: {search.best_score_:.4f}")
    print(f"Best params: {search.best_params_}")

    single_feature_tuning.append({
        "feature": feature,
        "best_cv_accuracy": search.best_score_,
        "best_params": search.best_params_
    })

results_df = pd.DataFrame(single_feature_tuning).sort_values(
    by="best_cv_accuracy", ascending=False
).reset_index(drop=True)

# Udvid best_params til kolonner, så det er nemmere at sammenligne
params_df = results_df["best_params"].apply(pd.Series)
results_pretty = pd.concat([
    results_df[["feature", "best_cv_accuracy"]],
    params_df
], axis=1)

print("Top 15 features med bedste grid-search parametre (5-fold CV):")
print(results_pretty.head(15))

best_row = results_df.iloc[0]
print("\nSamlet bedste feature + parametre:")
print(f"Feature: {best_row['feature']}")
print(f"Best CV accuracy: {best_row['best_cv_accuracy']:.4f}")
print(f"Best params: {best_row['best_params']}")

Klasse mapping:
BlueNoise -> 0
BrownNoise -> 1
Clean -> 2
PinkNoise -> 3
VioletNoise -> 4
WhiteNoise -> 5
Grid-kombinationer pr. feature: 72
Model fits pr. feature (kombinationer x 5 folds): 360

Feature: centroid_mean


In [ ]:
# ----------------------------
# 4. FEATURE IMPORTANCE
# ----------------------------
importances = model.feature_importances_
feature_names = X.columns

# Sorter features efter importance
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 8))
plt.title("Feature Importance (XGBoost)")
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), feature_names[indices], rotation=90)
plt.tight_layout()
plt.show()

# Print top 15 vigtigste features
print("\nTop 15 vigtigste features:")
for i in indices[:15]:
    print(f"{feature_names[i]}: {importances[i]:.4f}")

In [ ]:
# ----------------------------
# 5. Fjerne features baseret på importance
# ----------------------------

# Sæt en threshold for feature importance
importance_threshold = 0.02  # Juster denne værdi efter behov

# Find features der skal beholdes
features_to_keep = [feature for feature, importance in zip(feature_names, importances) if importance >= importance_threshold]
print(f"Beholder {len(features_to_keep)} features ud af {len(feature_names)} (threshold: {importance_threshold})")

# Fjern features med lav importance fra X
X_reduced = X[features_to_keep]
print("Features efter fjernelse:", X_reduced.columns.tolist())

# Nu kan X_reduced bruges til at træne en ny model eller til videre analyse

In [ ]:
# ----------------------------
# 6. ERROR ANALYSIS (SNR & BETA)
# ----------------------------

# gem predictions i dataframe
df_analysis = df_test.copy()
df_analysis["y_true"] = y_test_true
df_analysis["y_pred"] = y_test_pred
df_analysis["correct"] = df_analysis["y_true"] == df_analysis["y_pred"]

# behold kun støj (Clean har NaN)
df_noise = df_analysis.dropna(subset=["snr_db", "beta"]).copy()
# fejlrate pr SNR
snr_bins = np.arange(-5, 21, 2)
df_noise["snr_bin"] = pd.cut(df_noise["snr_db"], bins=snr_bins)

snr_error = df_noise.groupby("snr_bin")["correct"].mean()
snr_error = 1 - snr_error

plt.figure(figsize=(8,5))
snr_error.plot(kind="bar")
plt.title("XGBoost Model Error Rate vs SNR")
plt.ylabel("Error Rate")
plt.xlabel("SNR Bin (dB)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# fejlrate pr beta
beta_bins = np.linspace(-2.5, 2.5, 10)
df_noise["beta_bin"] = pd.cut(df_noise["beta"], bins=beta_bins)

beta_error = df_noise.groupby("beta_bin")["correct"].mean()
beta_error = 1 - beta_error

plt.figure(figsize=(8,5))
beta_error.plot(kind="bar")
plt.title("XGBoost Model Error Rate vs Beta")
plt.ylabel("Error Rate")
plt.xlabel("Beta Bin")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Lav prediction
y_test_pred = model.predict(X_test)

# Evaluer
print("\nTest Accuracy:", accuracy_score(y_test_true, y_test_pred))
print("\nClassification Report (Test):")
print(classification_report(y_test_true, y_test_pred, target_names=label_encoder.classes_))
print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test_true, y_test_pred))


# ----------------------------
# ERROR ANALYSIS
# ----------------------------

df_analysis = df_test.copy()

df_analysis["y_true"] = y_test_true
df_analysis["y_pred"] = y_test_pred
df_analysis["correct"] = df_analysis["y_true"] == df_analysis["y_pred"]

df_analysis["true_label"] = label_encoder.inverse_transform(df_analysis["y_true"])
df_analysis["pred_label"] = label_encoder.inverse_transform(df_analysis["y_pred"])

errors = df_analysis[df_analysis["correct"] == False]

print("\nEksempler på fejl:")
print(errors[["true_label","pred_label","beta","snr_db"]].head(20))